<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/autogent_human_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# may struggle for local setup highly recommended to run in vocarium
import autogen
print("AutoGen Installed Successfully:", autogen.__version__)


AutoGen Installed Successfully: 0.12.1


In [ ]:
# Import the Python SDK for Colab's secret manager
from google.colab import userdata

# Retrieve your API key from Colab Secrets
# Make sure you have added 'OPENAI_API_KEY' to your Colab Secrets
openai_api_key = userdata.get('OPENAI_API_KEY')

# Set the API key as an environment variable
os.environ["OPENAI_API_KEY"] = openai_api_key
print("OpenAI API Key loaded from Colab Secrets.")

OpenAI API Key loaded from Colab Secrets.


In [ ]:
# if not installed uncomment below 1 by 1 and install then restart kernel to test
!pip install autogen
!pip install ag2[openai]


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 14.4 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.31.0
    Uninstalling openai-2.31.0:
      Successfully uninstalled openai-2.31.0


In [ ]:
import os

# --- FIX: Set your OpenAI API key here or using Colab's Secrets feature ---
# If using Colab Secrets, enable 'OPENAI_API_KEY' in the 'Secrets' tab.
# If you prefer to set it directly (less secure for sharing), uncomment the line below
# and replace 'YOUR_OPENAI_API_KEY' with your actual key:
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

from autogen import ConversableAgent

agent = ConversableAgent(
    "chatbot",
    llm_config={
        "config_list": [
            {
                "model": "gpt-4o-mini",
                "api_key": os.environ.get("OPENAI_API_KEY")
            }
        ]
    },
    code_execution_config=False,  # Turn off code execution, by default it is off.
    function_map=None,  # No registered functions, by default it is None.
    human_input_mode="NEVER",  # Never ask for human input.
)

In [ ]:
reply = agent.generate_reply(messages=[{"content": "Tell me a joke.", "role": "user"}])
print(reply)

Why did the scarecrow win an award?

Because he was outstanding in his field!


In [ ]:
cathy = ConversableAgent(
    "cathy",
    system_message="Your name is Cathy and you are a part of a duo of comedians.",
    llm_config={"config_list": [{"model": "gpt-4o-mini", "temperature": 0.9, "api_key": os.environ.get("OPENAI_API_KEY")}]},
    human_input_mode="NEVER",  # Never ask for human input.
)

joe = ConversableAgent(
    "joe",
    system_message="Your name is Joe and you are a part of a duo of comedians.",
    llm_config={"config_list": [{"model": "gpt-4o-mini", "temperature": 0.7, "api_key": os.environ.get("OPENAI_API_KEY")}]},
    human_input_mode="NEVER",  # Never ask for human input.
)

In [ ]:
result = joe.initiate_chat(cathy, message="Cathy, tell me a joke.", max_turns=2)

joe (to cathy):

Cathy, tell me a joke.

--------------------------------------------------------------------------------
cathy (to joe):

Sure, Joe! Why don't scientists trust atoms? 

Because they make up everything!

--------------------------------------------------------------------------------
joe (to cathy):

Classic! Alright, here’s one for you: Why did the scarecrow win an award? 

Because he was outstanding in his field!

--------------------------------------------------------------------------------
cathy (to joe):

Cathy: That’s a good one! You know, I think scarecrows really know how to stand out—just like me at a silent meditation retreat!

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (13262f33-40a2-4b8a-8104-baf6e1a51f74): Maximum turns (2) reached


In [ ]:
#Allowing Human Feedback in Agents  via autogen

In [ ]:
agent_with_number = ConversableAgent(
    "agent_with_number",
    system_message="You are playing a game of guess-my-number. "
    "In the first game, you have the "
    "number 53 in your mind, and I will try to guess it. "
    "If I guess too high, say 'too high', if I guess too low, say 'too low'. ",
    llm_config={"config_list": [{"model": "gpt-4", "api_key": os.environ["OPENAI_API_KEY"]}]},
    max_consecutive_auto_reply=1,  # maximum number of consecutive auto-replies before asking for human input
    is_termination_msg=lambda msg: "53" in msg["content"],  # terminate if the number is guessed by the other agent
    human_input_mode="TERMINATE",  # ask for human input until the game is terminated
)

agent_guess_number = ConversableAgent(
    "agent_guess_number",
    system_message="I have a number in my mind, and you will try to guess it. "
    "If I say 'too high', you should guess a lower number. If I say 'too low', "
    "you should guess a higher number. ",
    llm_config={"config_list": [{"model": "gpt-4", "api_key": os.environ["OPENAI_API_KEY"]}]},
    human_input_mode="NEVER",
)

result = agent_with_number.initiate_chat(
    agent_guess_number,
    message="I have a number between 1 and 100. Guess it!",
)

agent_with_number (to agent_guess_number):

I have a number between 1 and 100. Guess it!

--------------------------------------------------------------------------------
agent_guess_number (to agent_with_number):

Is it 50?

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
agent_with_number (to agent_guess_number):

Too low.

--------------------------------------------------------------------------------
agent_guess_number (to agent_with_number):

Is it 75?

--------------------------------------------------------------------------------
Please give feedback to agent_guess_number. Press enter to skip and use auto-reply, or type 'exit' to stop the conversation: 10
agent_with_number (to agent_guess_number):

10

--------------------------------------------------------------------------------
agent_guess_number (to agent_with_number):

Apologies for the confusion. Instead of providing me with a number directly, you should hav

In [ ]:
# to know more visit https://microsoft.github.io/autogen/0.2/docs/tutorial